In [ ]:
import requests, json

SEARCH_URL = "https://civiweb-api-prd.azurewebsites.net/api/Offers/search"

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "*/*",
    "Origin": "https://mon-vie-via.businessfrance.fr",
    "Referer": "https://mon-vie-via.businessfrance.fr/",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "X-API-KEY": "l+KwpoLPiXlsjxNT/NQ2iOFz8+iuygxAODs9FeAEWYM=",
}

def base_payload(skip, limit=50):
    return {
        "limit": limit,
        "skip": skip,
        "sort": ["0"],
        "activitySectorId": [],
        "companiesSizes": [],
        "countriesIds": [],
        "entreprisesIds": [0],
        "geographicZones": [],
        "missionStartDate": None,
        "missionsDurations": [],
        "missionsTypesIds": [],
        "query": None,
        "specializationsIds": [],
        "studiesLevelId": [],
    }

def fetch_offers(limit=50, skip=0):
    offers = []
    while True:
        r = requests.post(SEARCH_URL, json=base_payload(skip, limit),
                          headers=HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()
        # ADJUST these keys once you paste the response:
        batch = data.get("result", [])
        total = data.get("count", 0)
        offers.extend(batch)
        skip += limit
        if skip >= total or not batch:
            break
    return offers

def find_week_old_offers_limit():
    offers, skip = [], 0
    while True:
        r = requests.post(SEARCH_URL, json=base_payload(skip, limit),
                            headers=HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()
        # ADJUST these keys once you paste the response:
        batch = data.get("result", [])
        total = data.get("count", 0)
        offers.extend(batch)
        skip += limit
        if skip >= total or not batch:
            break
    return offers

In [45]:
from datetime import date, timedelta, datetime
from zoneinfo import ZoneInfo

def first_offer(max_days=7):
    offers = {}
    today = datetime.now(ZoneInfo("Europe/Paris")).date()
    limit_date = today - timedelta(days=max_days)

    try:
        r = requests.post(SEARCH_URL, json=base_payload(skip=0, limit=1),
                                    headers=HEADERS, timeout=30)
        data = r.json()
    except Exception as e:
        raise ValueError(str(e))

    first_data = data.get('result',{})[0]
    first_offer_date = first_data.get("creationDate")

    offers["0"] = first_data

    try:
        last_offer_date = datetime.strptime(first_offer_date, "%Y-%m-%dT%H:%M:%SZ").date()
    except Exception as e:
        raise ValueError(f"first_offer_date problem, here is the first_offer's date - {first_offer_date}")
    
    return offers, limit_date, last_offer_date



def initialize_up_to_N_days(batch=10, max_days=7):
    offers, limit_date, last_offer_date = first_offer(max_days = max_days)


    while last_offer_date > limit_date:
        try:
            r = requests.post(SEARCH_URL, json=base_payload(skip=1, limit=batch),
                                        headers=HEADERS, timeout=30)
            data = r.json()
        except Exception as e:
            raise ValueError(str(e))

        results = data.get("result")
        print(len(results))
        print(json.dumps(results, indent=4))
        break

In [ ]:
import time
from tqdm import tqdm


T1 = time.time()
for i in tqdm(range(100)):
    try:
        r = requests.post(SEARCH_URL, json=base_payload(skip=0, limit=1),
                                                headers=HEADERS, timeout=30)
        data = r.json()
    except Exception as e:
        raise ValueError(str(e))
print(time.time() - T1)

In [ ]:
import time

T1 = time.time()
for _ in range(10):
    try:
        r = requests.post(SEARCH_URL, json=base_payload(skip=0, limit=10),
                                                headers=HEADERS, timeout=30)
        data = r.json()
    except Exception as e:
        raise ValueError(str(e))
print(time.time() - T1)

In [ ]:
initialize_up_to_N_days()

10
[
    {
        "id": 244624,
        "organizationName": "ENGINEERING & CONSULTING GROUP",
        "missionTitle": "Dessinateur Electrique (H/F)",
        "missionDuration": 12,
        "viewCounter": 0,
        "candidateCounter": 0,
        "missionType": "VIE",
        "missionTypeEn": "VIE",
        "organizationPresentation": "Vulcain Engineering, acteur majeur de l\u2019\u00e9nergie, est un groupe sp\u00e9cialis\u00e9 dans le conseil et l\u2019ing\u00e9nierie. Il a fait le choix d\u2019une expertise sectorielle dans l\u2019\u00e9nergie, l\u2019environnement, les sciences de la vie, l\u2019environnement et les infrastructures ferroviaire. Nous accompagnons nos clients dans leurs projets et leur transformation vers des mod\u00e8les plus responsables et durables, au service d\u2019une meilleure qualit\u00e9 de vie pour tous. Notre taille nous permet de garantir dans le m\u00eame temps force de frappe et proximit\u00e9 , nous sommes capables de mobiliser et d\u00e9ployer des ress